## Evaluation Protocol

Each benchmark follows the same pipeline:

1. **Corpus ingestion** — documents are chunked (if needed) and indexed by each adapter
2. **Query execution** — every query in the eval set is issued against the live index
3. **Ranking** — each adapter returns a ranked list of up to 10 results
4. **Scoring** — results are compared against ground-truth relevance labels using standard IR metrics

All adapters are evaluated against the same queries and the same relevance labels. No result is synthetic or interpolated.

```mermaid
flowchart LR
    C[Corpus] --> I[Index] --> Q[Query] --> R[Rank] --> S[Score]
```

## Metrics

### nDCG@10 (primary)

Normalized Discounted Cumulative Gain at rank 10 measures both whether relevant documents are retrieved and how highly they are ranked. A result at rank 1 contributes more than the same result at rank 10.

$$\text{nDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}}, \quad \text{DCG@k} = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i+1)}$$

Scores range from 0 (no relevant documents retrieved) to 1.0 (perfect ranking).

### Recall@k

Fraction of all relevant documents found in the top-k results. Reported at k=1, 5, 10.

### MRR@10

Mean Reciprocal Rank — the average of 1/rank for the first relevant result across all queries. Rewards putting a relevant result at the very top.

### Latency

Wall-clock query time measured after index warm-up (first query discarded). Reported as p50 and p95 across all queries in the eval set.

## Adapters

### SQLite FTS5

BM25 keyword search using SQLite's built-in full-text search extension. Documents are tokenised and stored in an FTS5 virtual table. Queries use OR-mode term matching. No embeddings are computed. Fast to index, zero external dependencies.

### LanceDB

Dense vector search using LanceDB with an HNSW index stored on disk. Documents are embedded with `all-MiniLM-L6-v2` at index time; queries are embedded at retrieval time. Cosine similarity is used for ranking.

### ChromaDB

Dense vector search using ChromaDB's embedded store. Same `all-MiniLM-L6-v2` embeddings as LanceDB. ChromaDB uses cosine similarity and stores its index in memory during a session.

### Tantivy

BM25 keyword search using Tantivy, a Rust-native full-text search library. Documents and queries are processed with Tantivy's default English analyser (stemming + stop-word removal). Indexed on disk.

## Embedding Model

Both dense-vector adapters use [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2):

- 384-dimensional embeddings
- 22M parameters
- Max sequence length: 256 tokens
- Optimised for semantic similarity

The same model version is used across all runs to ensure comparability.

## Reproducibility

All benchmark code, corpora preparation scripts, and raw result JSON files are in this repository. To reproduce a run:

```bash
mise run bench -- --benchmark <name> --store <adapter>
```

To run all benchmarks with all adapters:

```bash
mise run bench:fast
```

Results are written to `results/<benchmark>/<adapter>/<timestamp>.json`. The report is built from whatever results are present in `results/` at build time.

## Limitations

- **Single embedding model** — dense adapters share the same encoder; a different model would likely shift relative scores.
- **Small-scale corpora** — all four corpora fit in memory. Latency and recall characteristics may differ at larger scale.
- **English only** — no multilingual evaluation.
- **Retrieval only** — downstream generation quality (e.g., RAG answer correctness) is not measured.

### Pipeline Overview

```mermaid
flowchart LR
    subgraph Input
        C[(Corpus\ndocuments)]
        Q[(Query set\n+ qrels)]
    end
    C -->|chunk + embed| I[Index]
    Q -->|issue queries| E[Execute]
    I --> E
    E -->|ranked list k=10| M[Measure]
    M --> nDCG["nDCG@10\nR@k · MRR · Latency"]
    style Input fill:#f5f5f5,stroke:#ccc
    style nDCG fill:#e8f4e8,stroke:#888
```

### Search Architecture: BM25 vs Dense Vector

```mermaid
flowchart TB
    subgraph BM25["BM25 (SQLite FTS5 · Tantivy)"]
        direction TB
        Q1[Query text] -->|tokenise + stem| T1[Term list]
        T1 -->|inverted index lookup| S1["TF-IDF scores\n(BM25 formula)"]
        S1 --> R1[Ranked docs]
    end
    subgraph Dense["Dense Vector (LanceDB · ChromaDB · Qdrant)"]
        direction TB
        Q2[Query text] -->|encode: all-MiniLM-L6-v2| V2[384-dim vector]
        V2 -->|ANN: HNSW / cosine| S2[Similarity scores]
        S2 --> R2[Ranked docs]
    end
    style BM25 fill:#fff8e1,stroke:#f9a825
    style Dense fill:#e3f2fd,stroke:#1565c0
```

### nDCG@10: Position Matters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.set_facecolor('#fafafa'); fig.patch.set_facecolor('#fafafa')
for sp in ax.spines.values(): sp.set_visible(False)

ranks = np.arange(1, 11)
relevant = {1, 3, 7}  # hypothetical: relevant docs at positions 1, 3, 7
gains = [1.0 / np.log2(r + 1) if r in relevant else 0 for r in ranks]
ideal_gains = sorted(gains, reverse=True)

colors = ['#d45f5f' if r in relevant else '#d0d0d0' for r in ranks]
ax.bar(ranks, gains, color=colors, width=0.6, zorder=3)
ax.step(ranks, np.cumsum(ideal_gains), where='mid',
        color='#555', linestyle='--', linewidth=1.2, label='Ideal DCG (cumulative)')
ax.set_xlabel('Rank position', fontsize=10)
ax.set_ylabel('DCG gain  1/log₂(rank+1)', fontsize=10)
ax.set_title('DCG gain per rank — red bars = relevant documents', fontsize=10)
ax.set_xticks(ranks)
ax.legend(fontsize=9); ax.yaxis.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout(); plt.show()
print('nDCG@10  =  DCG@10 / IDCG@10   (normalised to [0, 1])')
print(f'Example: DCG@10 = {sum(gains):.3f},  IDCG@10 = {sum(ideal_gains):.3f},  nDCG@10 = {sum(gains)/sum(ideal_gains):.3f}')

### Recall@k: Coverage vs Depth

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt

ROOT = pathlib.Path().resolve()
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists(): ROOT = _p; break

adapters = ['sqlite', 'lancedb', 'chromadb', 'tantivy']
labels = {'sqlite': 'SQLite FTS5', 'lancedb': 'LanceDB', 'chromadb': 'ChromaDB', 'tantivy': 'Tantivy'}
colors = {'sqlite': '#f28e2b', 'lancedb': '#4e79a7', 'chromadb': '#59a14f', 'tantivy': '#e15759'}

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.set_facecolor('#fafafa'); fig.patch.set_facecolor('#fafafa')
for sp in ax.spines.values(): sp.set_visible(False)

for a in adapters:
    d = ROOT / 'results' / 'episodic-memory' / a
    fs = sorted(d.glob('*.json')) if d.exists() else []
    if not fs: continue
    r = json.loads(fs[-1].read_text())
    ks = [1, 5, 10]
    vals = [r.get(f'recall_at_{k}', 0) for k in ks]
    ax.plot(ks, vals, marker='o', color=colors[a], label=labels[a], linewidth=1.8)

ax.set_xlabel('k', fontsize=10); ax.set_ylabel('Recall@k', fontsize=10)
ax.set_title('Recall@k — Episodic Memory (increases monotonically with k)', fontsize=10)
ax.set_xticks([1, 5, 10]); ax.set_ylim(0, 1.05)
ax.legend(fontsize=9); ax.yaxis.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout(); plt.show()

### Reading the Metrics Together

| Metric | What it rewards | When to prioritise |
|--------|----------------|-------------------|
| **nDCG@10** | Highly-ranked relevant results | General retrieval quality |
| **R@1** | The very first result being correct | Single-shot answer generation |
| **R@10** | At least one hit anywhere in top 10 | RAG context window — any relevant chunk helps |
| **MRR@10** | The first relevant result appearing early | Interactive search — user sees top result first |
| **Latency p50** | Typical query speed | Real-time agent loops |
| **Latency p95** | Tail latency | SLA-sensitive deployments |

No metric tells the whole story. A RAG pipeline cares most about R@10 (fill the context window with anything useful) and latency. An agent that shows a single answer cares about R@1 and MRR. nDCG@10 is the standard headline metric because it balances precision and position.

## Adapter Algorithms

### BM25 (Best Match 25)

BM25 is the dominant keyword retrieval model. Given query terms $q_1 \ldots q_n$ and document $d$, it scores:

$$\text{BM25}(q,d) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i,d) \cdot (k_1 + 1)}{f(q_i,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

where $f(q_i, d)$ is term frequency, $|d|$ is document length, and $k_1$ (default 1.2) controls TF saturation while $b$ (default 0.75) controls length normalisation.

**Key intuition:** The TF component *saturates* — doubling the number of occurrences of a term in a document does not double its score. IDF gives rare terms more weight than common ones.

**Implementations here:**
- **SQLite FTS5** — uses FTS5's built-in BM25 scorer; no stemming unless configured
- **Tantivy** — English analyser: tokenise → lowercase → stop-word removal → Porter stemmer → BM25F scoring

> Robertson, S. & Zaragoza, H. (2009). *The probabilistic relevance framework: BM25 and beyond.* Foundations and Trends in Information Retrieval. [DOI:10.1561/1500000019](https://doi.org/10.1561/1500000019)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
fig.patch.set_facecolor('#fafafa')

# Left: TF saturation curve
ax = axes[0]; ax.set_facecolor('#fafafa')
for sp in ax.spines.values(): sp.set_visible(False)
tf = np.linspace(0, 20, 200)
k1 = 1.2
for b_val, ls in [(0.75, '-'), (0.5, '--'), (1.0, ':')]:
    dl_ratio = 1.0  # assume doc length = avg doc length
    denom = tf + k1 * (1 - b_val + b_val * dl_ratio)
    score = tf * (k1 + 1) / denom
    ax.plot(tf, score, linestyle=ls, linewidth=1.8, label=f'b={b_val}')
ax.set_xlabel('Term frequency f(q,d)', fontsize=9)
ax.set_ylabel('TF component score', fontsize=9)
ax.set_title(f'TF saturation (k₁={k1})', fontsize=9)
ax.legend(fontsize=8); ax.yaxis.grid(True, linestyle=':', alpha=0.5)

# Right: IDF weighting over corpus frequency
ax2 = axes[1]; ax2.set_facecolor('#fafafa')
for sp in ax2.spines.values(): sp.set_visible(False)
N = 1000  # total docs
df_vals = np.arange(1, N)
idf = np.log((N - df_vals + 0.5) / (df_vals + 0.5) + 1)
ax2.semilogx(df_vals, idf, color='#4e79a7', linewidth=1.8)
ax2.set_xlabel('Document frequency (log scale)', fontsize=9)
ax2.set_ylabel('IDF weight', fontsize=9)
ax2.set_title('IDF: rare terms get more weight', fontsize=9)
ax2.yaxis.grid(True, linestyle=':', alpha=0.5)

plt.suptitle('BM25 scoring components', fontsize=10, y=1.02)
plt.tight_layout(); plt.show()

### HNSW: Hierarchical Navigable Small World

HNSW (Hierarchical Navigable Small World) is the approximate nearest-neighbour (ANN) index used by LanceDB and Qdrant. It builds a multi-layer graph where upper layers have long-range connections and lower layers have dense local connections.

**Construction:** Each new vector is inserted into the graph at a randomly-selected maximum layer. Edges are added to the *M* nearest neighbours at each layer. At query time, a greedy search starts at the top layer (few nodes, fast coarse navigation) and descends to layer 0 (all nodes, fine refinement).

```mermaid
flowchart TB
    subgraph L2["Layer 2 (coarse)"]
        direction LR
        A2 --- B2 --- C2
    end
    subgraph L1["Layer 1"]
        direction LR
        A1 --- B1 --- C1 --- D1 --- E1
    end
    subgraph L0["Layer 0 (all vectors)"]
        direction LR
        A0 --- B0 --- C0 --- D0 --- E0 --- F0 --- G0
    end
    A2 --> A1; C2 --> D1
    A1 --> A0; D1 --> D0; E1 --> G0
    style L2 fill:#e3f2fd,stroke:#1565c0
    style L0 fill:#e8f5e9,stroke:#2e7d32
```

**Trade-offs:**
- `ef_construction` — more edges at build time → better recall, slower indexing
- `M` (connections per node) — higher M → higher recall, more memory
- `ef` (search beam) — wider beam at query time → higher recall, higher latency

> Malkov, Y. A., & Yashunin, D. A. (2018). *Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs.* IEEE TPAMI. [arXiv:1603.09320](https://arxiv.org/abs/1603.09320)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.set_facecolor('#fafafa'); fig.patch.set_facecolor('#fafafa')
for sp in ax.spines.values(): sp.set_visible(False)

ef_vals = np.array([10, 20, 50, 100, 200, 500])
# Illustrative recall curve (logistic-like, asymptotes at 1.0)
recall = 1 - np.exp(-ef_vals / 80)
latency = ef_vals * 0.04  # illustrative linear scaling

ax2 = ax.twinx()
ax.plot(ef_vals, recall, color='#4e79a7', marker='o', linewidth=1.8, label='Recall@10')
ax2.plot(ef_vals, latency, color='#e15759', marker='s', linewidth=1.8, linestyle='--', label='Latency (ms)')

ax.set_xlabel('ef (search beam width)', fontsize=9)
ax.set_ylabel('Recall@10 (illustrative)', fontsize=9, color='#4e79a7')
ax2.set_ylabel('Query latency ms (illustrative)', fontsize=9, color='#e15759')
ax.set_ylim(0, 1.05)
ax.set_title('HNSW: recall vs latency trade-off as ef increases', fontsize=9)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='lower right')
ax.yaxis.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout(); plt.show()

### SentenceTransformers: all-MiniLM-L6-v2

Both dense-vector adapters (LanceDB, ChromaDB) encode text using `sentence-transformers/all-MiniLM-L6-v2`. This model produces 384-dimensional embeddings optimised for semantic similarity via contrastive learning on a large corpus of sentence pairs.

**Architecture:** A siamese BERT-style network: two identical encoders share weights and process query/document independently; embeddings are compared via cosine similarity.

```mermaid
flowchart LR
    subgraph Encoder["Shared Encoder (all-MiniLM-L6-v2)"]
        direction TB
        T1["Query tokens"] --> B1[6-layer\nMiniLM] --> P1["Mean-pool"] --> E1["384-dim\nvector"]
        T2["Doc tokens"] --> B2[6-layer\nMiniLM] --> P2["Mean-pool"] --> E2["384-dim\nvector"]
    end
    E1 & E2 --> COS["cosine similarity\n\u2192 ranking score"]
    style Encoder fill:#fff8e1,stroke:#f9a825
```

**Training objective:** Contrastive learning — relevant pairs are pulled together in embedding space; irrelevant pairs are pushed apart. The resulting geometry means *semantically similar* sentences are close even if they share no vocabulary.

**Embedding properties:**
- 384 dimensions (compact; full BERT is 768)
- Max 256 tokens (longer docs are truncated; chunking mitigates this)
- Trained on 1B+ sentence pairs from diverse sources

> Reimers, N., & Gurevych, I. (2019). *Sentence-BERT: Sentence embeddings using Siamese BERT-networks.* EMNLP. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
>
> Wang, W. et al. (2020). *MiniLM: Deep self-attention distillation for task-agnostic compression of pre-trained transformers.* NeurIPS. [arXiv:2002.10957](https://arxiv.org/abs/2002.10957)

In [ ]:
# 2D PCA projection of a small sample of embeddings to illustrate the embedding space
# Uses synthetic example sentences so no model download is required at build time
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Synthetic clusters approximating what the embedding space looks like
categories = {
    'Storage / DB': (['WAL write-ahead log', 'checkpoint flush', 'page format', 'storage engine'], '#4e79a7'),
    'Consensus': (['Paxos quorum', 'Raft leader election', 'safekeeper replication', 'epoch commit'], '#e15759'),
    'Deployment': (['cloud deployment', 'container startup', 'health check endpoint', 'rolling update'], '#59a14f'),
    'API docs': (['POST request body', 'response schema', 'dependency injection', 'FastAPI router'], '#f28e2b'),
}

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.set_facecolor('#fafafa'); fig.patch.set_facecolor('#fafafa')
for sp in ax.spines.values(): sp.set_visible(False)

centers = [(-2, 2), (2, 2), (-2, -2), (2, -2)]
for (cat, (texts, color)), (cx, cy) in zip(categories.items(), centers):
    xs = rng.normal(cx, 0.35, len(texts))
    ys = rng.normal(cy, 0.35, len(texts))
    ax.scatter(xs, ys, color=color, s=80, label=cat, zorder=5)
    for x, y, t in zip(xs, ys, texts):
        ax.annotate(t, (x, y), xytext=(4, 3), textcoords='offset points', fontsize=6.5, color='#444')

ax.set_xlabel('PC1 (illustrative)', fontsize=9)
ax.set_ylabel('PC2 (illustrative)', fontsize=9)
ax.set_title('Embedding space: semantically similar texts cluster together\n(illustrative 2D PCA projection)', fontsize=9)
ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
plt.tight_layout(); plt.show()

## References

### Retrieval Models
1. Robertson, S. & Zaragoza, H. (2009). The probabilistic relevance framework: BM25 and beyond. *Foundations and Trends in Information Retrieval*, 3(4), 333–389. [doi:10.1561/1500000019](https://doi.org/10.1561/1500000019)
2. Malkov, Y. A., & Yashunin, D. A. (2018). Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs. *IEEE TPAMI*. [arXiv:1603.09320](https://arxiv.org/abs/1603.09320)
3. Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. *EMNLP 2019*. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
4. Wang, W. et al. (2020). MiniLM: Deep self-attention distillation for task-agnostic compression of pre-trained transformers. *NeurIPS 2020*. [arXiv:2002.10957](https://arxiv.org/abs/2002.10957)

### Benchmarks and Evaluation
5. Thakur, N. et al. (2021). BEIR: A heterogeneous benchmark for zero-shot evaluation of information retrieval models. *NeurIPS 2021 Datasets Track*. [arXiv:2104.08663](https://arxiv.org/abs/2104.08663)
6. Järvelin, K., & Kekäläinen, J. (2002). Cumulated gain-based evaluation of IR techniques. *ACM TOIS*, 20(4), 422–446. [doi:10.1145/582415.582418](https://doi.org/10.1145/582415.582418) *(nDCG definition)*

### Tools and Implementations
7. [sentence-transformers/all-MiniLM-L6-v2 — HuggingFace](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
8. [LanceDB Documentation](https://lancedb.github.io/lancedb/)
9. [ChromaDB Documentation](https://docs.trychroma.com/)
10. [Tantivy — Rust full-text search engine](https://github.com/quickwit-oss/tantivy)
11. [SQLite FTS5 Extension](https://www.sqlite.org/fts5.html)
12. [Qdrant Documentation](https://qdrant.tech/documentation/)